# ORCHAV Notebook Mode

Explore a Munich scene with a rooftop transmitter, a moving receiver, and stored propagation paths. From the repository root, install the notebook dependencies, generate the 100-frame recording, and start JupyterLab:

```bash
python -m pip install -e ".[jupyter]"
orchav-generator scenarios/visualizer/notebook_mode
jupyter lab --no-browser examples/notebook/notebook_mode.ipynb
```

On a remote machine, open the URL printed by JupyterLab in your browser. The cells read the recording in `scenarios/visualizer/notebook_mode`; rendering does not regenerate propagation paths. See [Notebook Visualization](../../docs/visualizer/notebooks.md) for frame selection, camera controls, and regeneration.

In [ ]:
from pathlib import Path

from visualizer.src.notebook.pygfx import PygfxNotebookViz


def find_repo_root(start: Path) -> Path:
    for path in (start.resolve(), *start.resolve().parents):
        if (path / "pyproject.toml").exists() and (path / "scenarios").is_dir():
            return path
    raise RuntimeError("Run this notebook from inside an ORCHAV checkout.")


REPO_ROOT = find_repo_root(Path.cwd())
SCENARIO = REPO_ROOT / "scenarios/visualizer/notebook_mode"
frames_dir = SCENARIO / "frames"
if not frames_dir.exists():
    raise RuntimeError(
        "Generate frames first with: orchav-generator scenarios/visualizer/notebook_mode"
    )

viz = PygfxNotebookViz(
    SCENARIO,
    project_root=REPO_ROOT,
    appearance="Dark",
    label_style="card",
)
frames = viz.frames
if not frames:
    viz.close()
    raise RuntimeError("No frames found. Generate the notebook-mode scenario first.")

print(viz)
print("frames:", frames)

VIEW = dict(
    center=(40, 90, 10),
    distance=220,
    azimuth=45,
    elevation=35,
    width=900,
    height=600,
)

## Interactive Playback

Play all 100 stored frames in a loop with the Dark appearance and TX/RX cards. Orbit, pan, and zoom remain available during playback, and TX/RX labels follow their markers. The requested rate of 24 frames per second gives a loop of about 4.2 s when the renderer maintains that rate. Playback speed is independent of the recording's 2 s simulation duration.

`jupyter_rfb` is included in the `jupyter` extra. Set `loop=False` to stop on the last frame, or call `animation.close()` to close this view.

The shared `VIEW` settings frame the TX/RX neighborhood. Adjust the camera interactively, or change these values to use the same framing in every view.

In [ ]:
animation = viz.animate(frames=frames, fps=24, loop=True, **VIEW)
animation

## Paper Appearance

Render the first available frame with the Paper appearance and propagation paths colored by reflection order. The four 3D presets are `Detailed`, `Paper`, `Dark`, `Glow`; pass `appearance` to any view to override the constructor default. Existing views keep their own appearance. `render()` displays the image inline and returns an `(H, W, 3)` NumPy array. It requires a working wgpu backend and does not open a desktop window.

In [ ]:
first_image = viz.render(
    appearance="Paper",
    frame=frames[0],
    color_mode="reflection_order",
    **VIEW,
    display=True,
)
first_image.shape

## Glow Appearance And Path-Loss Coloring

Render the last available frame with the Glow appearance from the same camera position. Explicit `color_mode="path_loss"` overrides the preset's delay metric. In the 100-frame recording, the receiver has reached the end of its route. Select indices from `viz.frames`; requesting an unavailable index raises `ValueError`.

In [ ]:
last_image = viz.render(
    appearance="Glow",
    frame=frames[-1],
    color_mode="path_loss",
    **VIEW,
    display=True,
)
last_image.shape

## Inspect A Still Frame

Inspect one frame with the Detailed appearance and TX/RX callouts. Change `label_style` to `"plain"` or `"card"` to compare labels. Marker sizes are in pixels; colors are RGB values between zero and one. Detailed uses installed textures and sky assets without downloading anything. `widget()` returns a canvas that can also be placed in an ipywidgets layout; `show()` provides the same orbit, pan, and zoom controls.

In [ ]:
canvas = viz.widget(
    frame=frames[0],
    appearance="Detailed",
    label_style={"mode": "callout", "tx_marker_size": 26, "rx_marker_size": 16},
    tx_color=(1.0, 0.35, 0.15),
    rx_color=(0.15, 0.65, 1.0),
    **VIEW,
)
canvas

## Close The Session

Uncomment the next cell when finished or before regenerating the recording externally. `close()` stops playback, closes this viewer's canvases, and releases its frame files. Static image outputs remain in the notebook.

After external generation finishes, call `viz.reload()` and refresh `frames = viz.frames`. Open a new widget or animation to view the regenerated recording.

In [ ]:
# viz.close()